In [ ]:
import pandas as pd
from collections import Counter
import re

In [ ]:
import warnings
warnings.simplefilter('ignore')

In [ ]:
COMSBSR = pd.read_csv('../../output/UNITED_2020-04-23.csv').\
    assign(comm_direct = lambda df: df['comm_parent_id'].str.contains('t3_')).\
    set_index(['subreddit','subm_id','comm_id'])
COMSBSR['comm_body'] = COMSBSR['comm_body'].astype(str)
COMSBSR.shape

In [ ]:
Counter(COMSBSR.comm_direct)

In [ ]:
from IPython.core.display import display, HTML

def highlight_re(text, regex):
    htmld = re.sub(r'('+regex+')', '<mark>' + r'\1' + '</mark>', text.lower())
    display(HTML(htmld))
    
def print_submission(i, index, row):
    print('\n\n\nr/%s \n--------------------' % (index[0]))
    print('Published: %s' % row.subm_created[:20])
    print('%d | SUBM: %10s | COMM: %10s\n--------------------' % (i, index[1], index[2]))
    highlight_re('<b>'+row.subm_title+'</b>', covid_regex)
    print('URL: %s\nSubmission score: %d' % (row.subm_url, row.subm_score))

def print_comment(i, index, row, flag_no, limit = 1000):
    print('\nFLAG #%d (score = %d)' % (flag_no, row.comm_score))
    highlight_re(row.comm_body[:limit], fake_regex)    

In [ ]:
covid_regex = '|'.join(pd.read_csv('../../input/keywords_covid.txt',header=None).loc[:,0].\
                  str.replace("\\(|\\)",r"").str.replace(r"\*",r""))#r"\\w*"))
fake_regex = '(fake |false).{0,15}(new\\w|info\\w|hood\\w)'+'|'+\
             '(dis|mis|mal)(info\\w|leading)'+'|'+\
             '\\bbs\\b|\\bbullshit\\b'+\
             '(not ).{0,5}(true\\b|verif\\w|real\\b)'+'|'+\
             '(unreli\\w+|unverifi\\w+|untrustw\\w+|fake\\w+)( source)'
fake_regex_no = '((no|not) .{0,10}('+fake_regex+'))'+'|(users often report submissions from this site)'

Extract comments with fake news regex

In [ ]:
highlight_re('some americans are misinformed about the coronavirus. how responsible is the media?', fake_regex)

In [ ]:
bool(re.search(fake_regex, 'misinformed'))

In [ ]:
COMM_WITH_FLAGS = COMSBSR.\
    query('comm_body.str.contains("'+fake_regex+'")').\
    query('not comm_body.str.contains("'+fake_regex_no+'")').\
    query('not subm_title.str.contains("'+fake_regex+'")').\
    query('comm_direct')
COMM_WITH_FLAGS.shape

In [ ]:
Counter(COMM_WITH_FLAGS.subm_title.str.contains('fake news'))

In [ ]:
COMM_WITH_FLAGS['subm_nflags'] = COMM_WITH_FLAGS.groupby(level = [0,1]).comm_body.transform('count')
#Counter(COMM_WITH_FLAGS['subm_nflags'])

In [ ]:
COMM_WITH_FLAGS_latest1000 = COMM_WITH_FLAGS.sort_values('subm_created',ascending=False)[:1000]
COMM_WITH_FLAGS_min3 = COMM_WITH_FLAGS.query('subm_nflags>2').sort_values('subm_created',ascending=False)
COMM_WITH_FLAGS_min3.shape

#### Print selection

In [ ]:
current_submission_id = 'none'
for i,(index,row) in enumerate(COMM_WITH_FLAGS_latest1000.iterrows()):
    if current_submission_id==index[1]:
        flag_no += 1
    else:
        print_submission(i,index,row)
        flag_no = 1
    current_submission_id=index[1]
    print_comment(i,index,row,flag_no)

In [ ]:
print('Percent of comments containing fake regex: %.2f' % (100*COMM_WITH_FLAGS.shape[0]/COMSBSR.shape[0])+'%')

In [ ]:
current_submission_id = 'none'
for i,(index,row) in enumerate(COMM_WITH_FLAGS_min3.iterrows()):
    if current_submission_id==index[1]:
        flag_no += 1
    else:
        print_submission(i,index,row)
        flag_no = 1
    current_submission_id=index[1]
    print_comment(i,index,row,flag_no)

### Non-POS flags

In [ ]:
import functions_pos

are flags

In [ ]:
subjects = ['you','it','this','that','article','title','headline','header']
predicates = ['be','spread','stop','call',\
              'look','sound','feel','seem','smell']
fullobjects = ['disinformation','misinformation','misleading','bullshit','bs','falsehood','fabrication']
partialobjects = ['news','information','title','source','article','header']
partialattributes = ['fake','false','misleading','untrustworthy','bogus','bullshit']
###
fake_regex_wrds = 'fake|false\\w*|bogus|'+\
                  'mislead\\w*|disinf\\w*|misinf\\w*|malinf\\w*|'+\
                  'bullshit|\\bbs\\b|'+\
                  'news|info\\w*|title|article|source|submission'

In [ ]:
are_flags = \
['how reliable is this source? ',
 'that’s the type of reactions this and many other misleading headlines do during times like these.',
 'this is how fake news is propagated.',
 'here comes the disinformation campaign',
 'the url sounds like one of those russian disinformation/dissension websites, doesn’t it?',
 'header is misleading.',
 'yeah cnn is a pretty unreliable source',
 'grade a example of fake news',
 'typical cnn fake news',
 "so i'm gonna guess this is a fake news article"]

In [ ]:
extract_pos(' '.join(are_flags))

In [ ]:
#Verb not root, but ccomp "guess/think this is fake news"

In [ ]:
[fake_sent_pos(nlp(s)) for s in are_flags]

are not flags

In [ ]:
are_not_flags = \
[
    'fake news! lol \*insert eye roll',
    'ppeople will still call this fake news. eople will still call this fake news.',
    'this is just fake news. /s',
    "experts? that's fake news. trump says it's going fine. ",  
]

In [ ]:
[fake_sent_pos(nlp(s)) for s in are_not_flags]